In [1]:
print('test')

test


In [4]:
import csv
import math
import time
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import xml.etree.ElementTree as ET

BASE_URL = "https://apis.data.go.kr/1390803/AgriFood/FdFood1/getKoreanFoodFdFoodList1"
SERVICE_KEY = "51cb7bbc7238b3a05c50974e40c97261a36015bddc473118eae5cc3c273094ce"

PAGE_SIZE = 20
SLEEP_SEC = 0.3


def text_or_none(elem):
    if elem is None or elem.text is None:
        return ""
    return elem.text.strip()


def find_first_text(root, names):
    for name in names:
        node = root.find(f".//{name}")
        if node is not None and node.text:
            return node.text.strip()
    return ""


def fetch_page(page_no, max_retries=5):
    params = {
        "serviceKey": SERVICE_KEY,
        "service_Type": "xml",
        "Page_No": page_no,
        "Page_Size": PAGE_SIZE,
    }

    url = BASE_URL + "?" + urlencode(params)

    for attempt in range(1, max_retries + 1):
        try:
            req = Request(
                url,
                headers={"User-Agent": "Mozilla/5.0"}
            )

            # 기존 30초 → 60초
            with urlopen(req, timeout=60) as resp:
                data = resp.read()

            root = ET.fromstring(data)

            result_code = find_first_text(
                root,
                ["result_Code", "resultCode"]
            )

            result_msg = find_first_text(
                root,
                ["result_Msg", "resultMsg"]
            )

            if result_code not in ("200", "0", "00"):
                raise RuntimeError(
                    f"API 오류 - page={page_no}, "
                    f"code={result_code}, msg={result_msg}"
                )

            total_count_text = find_first_text(
                root,
                ["total_Count", "totalCount"]
            )

            total_count = (
                int(total_count_text)
                if total_count_text else None
            )

            items = root.findall(".//items/item")

            return total_count, items

        except Exception as e:

            print(
                f"\n[재시도 {attempt}/{max_retries}] "
                f"page={page_no} 오류: {e}"
            )

            if attempt == max_retries:
                raise

            # 2초 → 4초 → 6초 → 8초
            wait_sec = attempt * 2

            print(f"{wait_sec}초 후 다시 요청합니다.")
            time.sleep(wait_sec)


def direct_child_dict(item):
    row = {}
    for child in list(item):
        if child.tag == "food_List":
            continue
        if len(list(child)) == 0:
            row[child.tag] = text_or_none(child)
    return row


def ingredient_dict(food):
    row = {}
    for child in list(food):
        if len(list(child)) == 0:
            row[child.tag] = text_or_none(child)
    return row


def write_csv(path, rows):
    if not rows:
        print(f"[경고] 저장할 데이터 없음: {path}")
        return

    fieldnames = []
    seen = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                fieldnames.append(key)

    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def main():
    print("메뉴젠 전체 데이터 수집 시작")

    total_count, first_items = fetch_page(1)
    if total_count is None:
        raise RuntimeError("total_Count를 찾지 못했습니다.")

    total_pages = math.ceil(total_count / PAGE_SIZE)
    print(f"전체 메뉴 수: {total_count:,}")
    print(f"페이지 수: {total_pages} (페이지당 {PAGE_SIZE}건)")

    menu_rows = []
    ingredient_rows = []

    for page_no in range(1, total_pages + 1):
        if page_no == 1:
            items = first_items
        else:
            _, items = fetch_page(page_no)

        for item in items:
            menu = direct_child_dict(item)
            menu_rows.append(menu)

            food_list = item.find("food_List")
            if food_list is not None:
                for food in food_list.findall("food"):
                    ing = ingredient_dict(food)

                    combined = {
                        "menu_fd_Code": menu.get("fd_Code", ""),
                        "menu_fd_Nm": menu.get("fd_Nm", ""),
                        "menu_upper_Fd_Group_Nm": menu.get("upper_Fd_Group_Nm", ""),
                        "menu_fd_Group_Nm": menu.get("fd_Group_Nm", ""),
                        "menu_fd_Wgh": menu.get("fd_Wgh", ""),
                    }
                    for k, v in ing.items():
                        combined[f"ingredient_{k}"] = v

                    ingredient_rows.append(combined)

        print(
            f"[{page_no:>2}/{total_pages}] "
            f"메뉴 누적 {len(menu_rows):,} / "
            f"식재료행 누적 {len(ingredient_rows):,}"
        )
        time.sleep(SLEEP_SEC)

    write_csv("menugen_menu_master.csv", menu_rows)
    write_csv("menugen_menu_ingredients.csv", ingredient_rows)

    unique_menu_codes = {r.get("fd_Code", "") for r in menu_rows if r.get("fd_Code")}
    unique_menu_names = {r.get("fd_Nm", "") for r in menu_rows if r.get("fd_Nm")}
    unique_ingredient_names = {
        r.get("ingredient_food_Nm", "")
        for r in ingredient_rows
        if r.get("ingredient_food_Nm")
    }

    print("\n=== 수집 완료 ===")
    print(f"API total_Count        : {total_count:,}")
    print(f"수집 메뉴 행           : {len(menu_rows):,}")
    print(f"고유 메뉴 코드          : {len(unique_menu_codes):,}")
    print(f"고유 메뉴명             : {len(unique_menu_names):,}")
    print(f"식재료 행               : {len(ingredient_rows):,}")
    print(f"고유 식재료명           : {len(unique_ingredient_names):,}")
    print("\n생성 파일")
    print(" - menugen_menu_master.csv")
    print(" - menugen_menu_ingredients.csv")


if __name__ == "__main__":
    main()


메뉴젠 전체 데이터 수집 시작
전체 메뉴 수: 3,250
페이지 수: 163 (페이지당 20건)
[ 1/163] 메뉴 누적 20 / 식재료행 누적 39
[ 2/163] 메뉴 누적 40 / 식재료행 누적 105
[ 3/163] 메뉴 누적 60 / 식재료행 누적 227
[ 4/163] 메뉴 누적 80 / 식재료행 누적 289
[ 5/163] 메뉴 누적 100 / 식재료행 누적 356
[ 6/163] 메뉴 누적 120 / 식재료행 누적 442
[ 7/163] 메뉴 누적 140 / 식재료행 누적 600
[ 8/163] 메뉴 누적 160 / 식재료행 누적 784
[ 9/163] 메뉴 누적 180 / 식재료행 누적 1,032
[10/163] 메뉴 누적 200 / 식재료행 누적 1,251
[11/163] 메뉴 누적 220 / 식재료행 누적 1,462
[12/163] 메뉴 누적 240 / 식재료행 누적 1,671
[13/163] 메뉴 누적 260 / 식재료행 누적 1,903
[14/163] 메뉴 누적 280 / 식재료행 누적 2,144
[15/163] 메뉴 누적 300 / 식재료행 누적 2,372
[16/163] 메뉴 누적 320 / 식재료행 누적 2,604
[17/163] 메뉴 누적 340 / 식재료행 누적 2,833
[18/163] 메뉴 누적 360 / 식재료행 누적 3,025
[19/163] 메뉴 누적 380 / 식재료행 누적 3,236
[20/163] 메뉴 누적 400 / 식재료행 누적 3,296
[21/163] 메뉴 누적 420 / 식재료행 누적 3,317
[22/163] 메뉴 누적 440 / 식재료행 누적 3,348
[23/163] 메뉴 누적 460 / 식재료행 누적 3,533
[24/163] 메뉴 누적 480 / 식재료행 누적 3,568
[25/163] 메뉴 누적 500 / 식재료행 누적 3,588
[26/163] 메뉴 누적 520 / 식재료행 누적 3,608
[27/163] 메뉴 누적 540 / 식재료행 누적 3,758
[28/163] 메뉴 누적 560 / 식